In [2]:
import random
from pokerkit import *
from dotenv import load_dotenv
import json
from anthropic import Anthropic, beta_tool

In [3]:
load_dotenv()

client = Anthropic()

In [25]:
system_prompt = """You are a poker agent playing 3-handed Texas Hold'em No-Limit. 
Blinds are 1000/2000. You are Player 1 (Hero).
Think about pot odds, position, and hand strength before deciding. Make sure to carefully read and use the information in the provided json
Use the take_action tool to submit your decision."""

@beta_tool
def take_action(action: str, reasoning: str, amount: float = 0) -> str:
    """Submit your poker action for this turn and give a maximum 3 sentence explanation for why.

    Args:
        action: One of fold, check, call, raise
        amount: Raise amount in chips. Required if action is raise.
        reasoning: Reasoning for why you took a poker action. Maximum 3 sentences. Rate how aggressive each opponent's actions are on a 1-10 scale. 
    """
    return json.dumps({"status": "accepted", "action": action, "amount": amount, 'reasoning': reasoning})

def run_turn(state_json):
    messages = [{
        "role": "user",
        "content": f"""It's your turn to act. Here is the current game state:
    
    ```json
    {state_json}
    ```
    
    Decide your action."""}]

    runner = client.beta.messages.tool_runner(
        model="claude-haiku-4-5-20251001",
        max_tokens=1024,
        tools=[take_action],
        system=system_prompt,
        messages=messages
    )

    agent_action = None

    for message in runner:
        tokens = f"Input: {message.usage.input_tokens}, Output: {message.usage.output_tokens}"

        for block in message.content:
            if block.type == "tool_use" and block.name == "take_action":
                agent_action = block.input

    return agent_action, tokens

def street_converter(index):
    if index == 0:
        return 'preflop'
    elif index == 1:
        return 'flop'
    elif index == 2:
        return 'turn'
    elif index == 3:
        return 'river'
    
def get_visible_operations(state):
    visible = []
    for op in state.operations:
        if isinstance(op, HoleDealing):
            continue
        elif isinstance(op, CardBurning):
            # Never visible
            continue
        else:
            visible.append(op)
    return visible


In [26]:
# Full game

# Initial Setup
player_stacks = [10_000, 10_000, 10_000] # Starting stacks for 3 players [cite: 85]
hand_count = 0

# The Global Game Loop: Runs until only one player has chips
while len([s for s in player_stacks if s > 0]) > 1:
    reasonings = []
    active_indices = [i for i, stack in enumerate(player_stacks) if stack > 0]

    # If only one player is left, the game is over
    if len(active_indices) < 2:
        break

# Filter stacks for the engine
    current_stacks = tuple(player_stacks[i] for i in active_indices)
    hand_count += 1
    print(f"\n--- STARTING HAND {hand_count} ---")
    
    state = NoLimitTexasHoldem.create_state(
        (
            Automation.ANTE_POSTING,
            Automation.BET_COLLECTION,
            Automation.BLIND_OR_STRADDLE_POSTING,
            Automation.CARD_BURNING,
            Automation.HOLE_DEALING,
            Automation.BOARD_DEALING,
            Automation.HOLE_CARDS_SHOWING_OR_MUCKING,
            Automation.HAND_KILLING,
            Automation.CHIPS_PUSHING,
            Automation.CHIPS_PULLING,
        ),
        True, 0, (1000, 2000), 2000, tuple(current_stacks), len(active_indices)
    )
    #big blind ante, ante, blinds, min bet, stacks, players

    # Internal Decision Loop (Per-Turn) [cite: 19, 49]
    while state.status:
        if state.can_deal_hole():
            state.deal_hole()
        elif state.can_deal_board():
            state.deal_board()
            
        elif state.actor_index is not None:
            res = get_visible_operations(state)
            obs = {
                "your_index": 0,
                "pot": state.total_pot_amount,
                "board": state.board_cards,
                "hole": state.hole_cards[state.actor_index],
                "stacks": state.stacks,
                "bets": state.bets,
                "street": street_converter(state.street_index),
                "can_fold?": state.can_fold(),
                "can_check_or_call?": state.can_check_or_call(),
                "can_raise?": state.can_complete_bet_or_raise_to(),
                "min_raise": state.min_completion_betting_or_raising_to_amount,
                "max_raise": state.max_completion_betting_or_raising_to_amount,
                "how_much_to_call": state.checking_or_calling_amount,          # how much a call costs
                "action_history": res
            }
            if state.actor_index == 0: # Agent
                # print(obs)
                action = run_turn(obs)
                if action[0]['action'] in ['check', 'call']:
                    state.check_or_call() 
                elif action[0]['action'] == 'raise':
                    state.complete_bet_or_raise_to(action[0]['amount'])
                elif action[0]['action'] == 'fold':
                    state.fold()
                try:
                    reasonings.append([action[0]['action'], action[0]['amount'], action[0]['reasoning']])
                except:
                    reasonings.append([action[0]['action'], 0, action[0]['reasoning']])
            else: 
                action_rand = random.random()
                if action_rand < 0.1 and state.can_complete_bet_or_raise_to():
        # 10% chance to raise to 3x the big blind
                    min_raise = state.min_completion_betting_or_raising_to_amount
                    max_raise = state.max_completion_betting_or_raising_to_amount # All-in
            
            # Raise to 2x the minimum required to keep it moving
                    target_raise = min(min_raise * 2, max_raise)
                    state.complete_bet_or_raise_to(target_raise)
                elif action_rand < 0.2 and state.can_fold():
        # 10% chance to fold
                    state.fold()
                else:
                    state.check_or_call()
        else:
            break

    # Update player stacks for the next hand based on payoffs [cite: 91, 92]
    # payoffs return the net change (e.g., [-2000, 4000, -2000])
    print(list(map(lambda x: str(x), get_visible_operations(state))))
    print(reasonings)
    for i, global_idx in enumerate(active_indices):
        player_stacks[global_idx] += int(state.payoffs[i])
    
    print(f"Hand {hand_count} Results: {state.payoffs}")
    print(f"New Stacks: {player_stacks}")

print(f"\nGAME OVER after {hand_count} hands!")
print(f"Final Winner Stacks: {player_stacks}")


--- STARTING HAND 1 ---
['BlindOrStraddlePosting(commentary=None, player_index=0, amount=1000)', 'BlindOrStraddlePosting(commentary=None, player_index=1, amount=2000)', 'CheckingOrCalling(commentary=None, player_index=2, amount=2000)', 'Folding(commentary=None, player_index=0)', 'CheckingOrCalling(commentary=None, player_index=1, amount=0)', 'BetCollection(commentary=None, bets=(1000, 2000, 2000))', 'BoardDealing(commentary=None, cards=(8h, Ah, 3c))', 'CheckingOrCalling(commentary=None, player_index=1, amount=0)', 'CheckingOrCalling(commentary=None, player_index=2, amount=0)', 'BoardDealing(commentary=None, cards=(2d,))', 'CheckingOrCalling(commentary=None, player_index=1, amount=0)', 'CompletionBettingOrRaisingTo(commentary=None, player_index=2, amount=4000)', 'CheckingOrCalling(commentary=None, player_index=1, amount=4000)', 'BetCollection(commentary=None, bets=(0, 4000, 4000))', 'BoardDealing(commentary=None, cards=(8s,))', 'CheckingOrCalling(commentary=None, player_index=1, amount

KeyboardInterrupt: 

In [17]:
res

[BlindOrStraddlePosting(commentary=None, player_index=0, amount=1000),
 BlindOrStraddlePosting(commentary=None, player_index=1, amount=2000),
 CheckingOrCalling(commentary=None, player_index=2, amount=2000)]